# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**My rule idea, in plain words:**
> Review **young pages** whose keyword has **low search volume** first. Among those, put the pages
> with the most February impressions at the top, because they have the most visibility to lose.

- **Reason code:** `young_low_volume`
- **Action label:** `review_for_decline_risk`

**Why these two signals:** in Week 2 (starter CSV), younger pages declined more (gap −16.4 points)
and pages with lower search volume declined more (gap −12.2). **Search volume** is also the signal
behind FlyRank's **quick-win** flag. I do not use staleness (days since the last update): in
Week 3, 76.5% of pages were updated after 1 March, so that date is not known on the decision day.

**Data:** the same Feb → Mar 2026 warehouse frame as Week 3 (features from 1–28 Feb, outcome from
March, same filters). The model in Week 5 will use the same pages and labels, so the comparison is fair.

**Before I build the rule, I check both signals.** Each gets a table: pages split into buckets,
with the number of pages (n) and the % declining in each bucket.

**How I give the verdict (decided before looking):** I only count buckets with at least 500 pages.
- **CONFIRMED:** decline goes the way my rule expects (younger / lower volume → more decline) and
  the two end buckets differ by 5+ points.
- **OPPOSITE:** the two end buckets differ by 5+ points, but the other way.
- **MIXED:** the pattern goes up and down, or the difference is only 2–5 points.
- **FALSE:** every bucket is within 2 points of the base rate.

In [ ]:
# Setup: connect DuckDB to the warehouse (token comes from the Colab Secrets panel, never typed here)
%pip -q install duckdb
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Only the two months I need (a list of paths; hf:// doesn't support {a,b} globs)
FACT_FEB_MAR = (f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
                f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])")

BEFORE_START, DECISION, AFTER_END = "2026-02-01", "2026-03-01", "2026-03-31"
print('Connected to the warehouse.')

In [ ]:
FEATURES = ["content_age_days", "word_count", "search_volume", "feb_avg_position", "feb_impressions"]

# Build one row per page: Feb features + March outcome
frame = con.sql(f"""
    WITH ok_clients AS (
        SELECT client_hash_id FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{BEFORE_START}'
    ),
    per_page AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_impressions,
               SUM(f.gsc_avg_position * f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}')
                 / NULLIF(SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}'), 0)
                                                                                         AS feb_avg_position,
               COUNT(*) FILTER (WHERE f.report_date >= DATE '{DECISION}')                AS mar_days,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date >= DATE '{DECISION}')  AS mar_impressions
        FROM {FACT_FEB_MAR} f
        JOIN ok_clients USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.content_hash_id,
           date_diff('day', d.content_created_date, DATE '{DECISION}') AS content_age_days,
           d.word_count, d.search_volume,
           p.feb_avg_position, p.feb_impressions, p.mar_days, p.mar_impressions
    FROM per_page p
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE d.content_created_date < DATE '{BEFORE_START}'
      AND p.feb_impressions >= 100
""").df()

pages_before = len(frame)
frame = frame[frame["mar_days"] > 0].copy()          # pages with no usable March rows: label unknown
dropped_no_march = pages_before - len(frame)

# Label: March impressions per day more than 20% below February's
feb_per_day = frame["feb_impressions"] / 28
mar_per_day = frame["mar_impressions"] / 31
frame["is_declining"] = (mar_per_day < 0.8 * feb_per_day).astype(int)

print(f"Pages in the frame: {len(frame):,} from {frame['client_hash_id'].nunique()} clients")
print(f"Dropped (no usable March data): {dropped_no_march:,}")
print(f"Base rate: {frame['is_declining'].mean() * 100:.1f}% declining\n")
print("Feature summary (no rows shown):")
print(frame[FEATURES].astype(float).describe().T[["count", "mean", "50%", "min", "max"]].round(1).to_string())
print("\nMissing per feature (%):")
print((frame[FEATURES].isna().mean() * 100).round(1).to_string())

### Signal 1: page age (my rule expects: younger → more decline)

**My verdict (write after Run all):** _CONFIRMED / OPPOSITE / MIXED / FALSE, and why in one line._

In [ ]:
# Bucket table helper: n and % declining per bucket, plus missing values
def bucket_table(col, bins, labels):
    buckets = pd.cut(frame[col], bins=bins, labels=labels, right=False)
    t = frame.groupby(buckets, observed=False)["is_declining"].agg(pages="size", decline_rate="mean")
    missing = frame[col].isna()
    if missing.any():
        t.loc["missing"] = [missing.sum(), frame.loc[missing, "is_declining"].mean()]
    t["pages"] = t["pages"].astype(int)
    t["decline_%"] = (t.pop("decline_rate") * 100).round(1)
    t["vs_base_pts"] = (t["decline_%"] - BASE).round(1)
    return t

def read_table(t):
    big = t.drop(index="missing", errors="ignore")
    big = big[big["pages"] >= 500]
    if len(big) < 2:
        print("\nFewer than 2 buckets have 500+ pages: too small to judge.")
        return
    first, last = big["decline_%"].iloc[0], big["decline_%"].iloc[-1]
    steps = big["decline_%"].diff().dropna()
    one_way = (steps <= 0).all() or (steps >= 0).all()
    print(f"\nBuckets with 500+ pages: {len(big)}")
    print(f"First bucket {first}% -> last bucket {last}%: difference {last - first:+.1f} points")
    print("Goes one way (no ups and downs):", one_way)
    print("Biggest distance from base rate:", f"{big['vs_base_pts'].abs().max():.1f} points")

BASE = frame["is_declining"].mean() * 100
print(f"Base rate: {BASE:.1f}% declining ({len(frame):,} pages)\n")

age = bucket_table("content_age_days",
                   bins=[0, 90, 180, 270, 365, float("inf")],
                   labels=["<90 days", "90-179", "180-269", "270-364", "365+"])
print("Decline by page age (days on 1 March):")
print(age.to_string())
read_table(age)

### Signal 2: search volume (flag-linked: quick-win). My rule expects: lower volume → more decline

**My verdict (write after Run all):** _CONFIRMED / OPPOSITE / MIXED / FALSE, and why in one line._

In [ ]:
volume = bucket_table("search_volume",
                      bins=[0, 1, 11, 101, 1001, float("inf")],
                      labels=["0", "1-10", "11-100", "101-1000", "1001+"])
print("Decline by keyword search volume:")
print(volume.to_string())
read_table(volume)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.